Chapter 4 Exercise solutions

Exercise 4.1: Parameters in the feed forward versus attention module

In [ ]:
# 中文说明：本单元用于检查当前环境安装的 PyTorch 版本，为后续构建/统计 GPT 模型参数量做准备。
from importlib.metadata import version

# 风险提示（仅标注，不修改）：下面这行从 fastai 库导入了 total_params 工具函数，
# 但整份笔记本后续统计参数量时，实际用的都是 `sum(p.numel() for p in ...parameters())` 手动计算，
# 并没有调用这里导入的 total_params，这个导入看起来是多余/无关的遗留代码。
# 如果当前环境没有安装 fastai 库，这一行会直接抛出 ImportError 导致本单元执行失败；
# 这不是本章知识点相关的导入，运行前请确认已安装 fastai，或按需自行移除此行。
from fastai.callback.hook import total_params

print("torch version:", version("torch"))  # 中文：打印当前 torch 版本号，便于环境核对


In [ ]:
# 中文说明：本单元对应习题 4.1 ——比较前馈网络（FeedForward）模块与
# 多头注意力（Attention）模块各自的参数量。
# 这里先从 gpt.py 中导入 TransformerBlock，并使用标准 GPT-2 small（124M）配置
# 构造一个单独的 Transformer 块，供下面两个单元分别统计其中 ff 子模块与 att 子模块的参数量。
from gpt import TransformerBlock
GPT_CONFIG_124M={
     "vocab_size": 50257,     # 中文：词表大小（BPE 分词器的词汇量）
    "context_length": 1024,   # 中文：模型支持的最大上下文长度（位置嵌入的行数）
    "emb_dim": 768,           # 中文：每个 token 的嵌入/隐藏层维度
    "n_heads": 12,            # 中文：多头注意力的头数；每个头维度 head_dim = 768/12 = 64
    "n_layers": 12,           # 中文：堆叠的 TransformerBlock 层数（这里只构造 1 个用于分析）
    "drop_rate": 0.1,         # 中文：统一的 dropout 概率（本单元用旧版单一 drop_rate 配置）
    "qkv_bias": False         # 中文：计算 Q/K/V 的线性层不使用偏置项
}
block=TransformerBlock(GPT_CONFIG_124M)
print(block)  # 中文：打印该 Transformer 块的结构，可以看到内部包含 att、ff、norm1、norm2、drop_shortcut 等子模块


In [ ]:
# 中文说明：统计 block.ff（FeedForward 前馈网络子模块）的参数量。
# FeedForward 结构为 Linear(emb_dim, 4*emb_dim) -> GELU -> Linear(4*emb_dim, emb_dim)，两层都带 bias。
# 参数量 = (emb_dim*4*emb_dim + 4*emb_dim) + (4*emb_dim*emb_dim + emb_dim)
#        = 768*3072 + 3072 + 3072*768 + 768 = 4,722,432（emb_dim=768 时）
total_params=sum(p.numel() for p in block.ff.parameters())
print(f"Total number of parameters in feed forward module: {total_params:,}")


In [ ]:
# 中文说明：统计 block.att（MultiHeadAttention 多头注意力子模块）的参数量，与上一单元的 ff 参数量做对比。
# 注意力模块包含 W_query / W_key / W_value 三个 Linear(emb_dim, emb_dim, bias=False)，
# 以及一个 out_proj = Linear(emb_dim, emb_dim)（默认带 bias）。
# 参数量 = 3*(emb_dim*emb_dim) + (emb_dim*emb_dim + emb_dim)
#        = 3*768*768 + 768*768 + 768 = 2,360,064（emb_dim=768 时）
# 可以看到 FeedForward 的参数量（约 472 万）大约是 Attention 模块参数量（约 236 万）的 2 倍，
# 这正是习题 4.1 想说明的结论：GPT 模型中前馈网络占用的参数远多于注意力模块。
total_params = sum(p.numel() for p in block.att.parameters())
print(f"Total number of parameters in attention module: {total_params:,}")


Exercise 4.2: Initialize larger GPT models
GPT2-small (the 124M configuration we already implemented):

"emb_dim" = 768
"n_layers" = 12
"n_heads" = 12
GPT2-medium:

"emb_dim" = 1024
"n_layers" = 24
"n_heads" = 16
GPT2-large:

"emb_dim" = 1280
"n_layers" = 36
"n_heads" = 20
GPT2-XL:

"emb_dim" = 1600
"n_layers" = 48
"n_heads" = 25

In [ ]:
# 中文说明：本单元对应习题 4.2 ——初始化不同规模的 GPT-2 模型（small/medium/large/xl），
# 并封装两个工具函数：
#   1) get_config：根据模型名称覆盖基础配置中的 emb_dim / n_layers / n_heads，得到对应规模的配置字典；
#   2) calculate_size：给定一个已构造好的模型，统计其参数量和显存/磁盘占用大小。
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}
def get_config(base_config, model_name="gpt2-small"):
    GPT_CONFIG = base_config.copy()  # 中文：复制一份基础配置，避免直接修改传入的原字典

    if model_name == "gpt2-small":
        GPT_CONFIG["emb_dim"] = 768
        GPT_CONFIG["n_layers"] = 12
        GPT_CONFIG["n_heads"] = 12

    elif model_name == "gpt2-medium":
        GPT_CONFIG["emb_dim"] = 1024
        GPT_CONFIG["n_layers"] = 24
        GPT_CONFIG["n_heads"] = 16

    elif model_name == "gpt2-large":
        GPT_CONFIG["emb_dim"] = 1280
        GPT_CONFIG["n_layers"] = 36
        GPT_CONFIG["n_heads"] = 20

    elif model_name == "gpt2-xl":
        GPT_CONFIG["emb_dim"] = 1600
        GPT_CONFIG["n_layers"] = 48
        GPT_CONFIG["n_heads"] = 25

    else:
        raise ValueError(f"Incorrect model name {model_name}")

    return GPT_CONFIG
def calculate_size(model):
    # 中文：模型的原始（未去重）参数总量，即 model.parameters() 里所有张量元素个数之和。
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total number of parameters: {total_params:,}")
    # 中文：原始 GPT-2 论文中，输出层 out_head 与词嵌入 tok_emb 是权重共享（weight tying）的，
    # 也就是说 out_head 的参数其实不需要单独存储/训练。这里的实现并没有真正做权重绑定，
    # 而是用「总参数量 - out_head 参数量」来近似模拟权重共享后的可训练参数量，方便与官方 GPT-2 报告的参数量对齐。
    total_params_gpt2=total_params-sum(p.numel() for p in model.out_head.parameters())
    print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")
    # Calculate the total size in bytes (assuming float32, 4 bytes per parameter)
    # 中文：假设每个参数用 float32 存储，每个参数占 4 字节，估算模型占用的显存/磁盘大小。
    total_size_bytes = total_params * 4

    # Convert to megabytes
    # 中文：字节数转换为 MB（1 MB = 1024*1024 字节）。
    total_size_mb = total_size_bytes / (1024 * 1024)

    print(f"Total size of the model: {total_size_mb:.2f} MB")


In [ ]:
# 中文说明：依次构造 gpt2-small / medium / large / xl 四种规模的 GPTModel，
# 并调用上面定义的 calculate_size 打印每个模型的参数量与占用大小，直观对比模型规模差异。
# 注意：n_layers 越大、emb_dim 越大，模型参数量会显著增长（大致与 emb_dim 的平方成正比，
# 因为注意力和前馈网络里的线性层参数量都正比于 emb_dim^2）。
from gpt import GPTModel
for model_abbrev in ("small", "medium", "large", "xl"):
    model_name = f"gpt2-{model_abbrev}"
    CONFIG = get_config(GPT_CONFIG_124M, model_name=model_name)  # 中文：得到该规模对应的配置字典
    model = GPTModel(CONFIG)  # 中文：按配置构造完整 GPT 模型（词嵌入+位置嵌入+N层Block+输出头）
    print(f"\n\n{model_name}:")
    calculate_size(model)


Exercise 4.3: Using separate dropout parameters

In [ ]:
# 中文说明：本单元对应习题 4.3 ——将原来单一的 drop_rate 拆分成三个独立的 dropout 概率，
# 分别控制词嵌入层、多头注意力、残差（shortcut）连接处的 dropout，
# 这样可以更精细地为模型的不同部分设置不同的正则化强度（原书默认三者都用同一个 0.1）。
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate_emb": 0.1,        # NEW: dropout for embedding layers
    "drop_rate_attn": 0.1,       # NEW: dropout for multi-head attention
    "drop_rate_shortcut": 0.1,   # NEW: dropout for shortcut connections
    "qkv_bias": False
}


In [ ]:
# 中文说明：重新定义 TransformerBlock 与 GPTModel，使用上一单元中拆分出的三个独立 dropout 参数
# （drop_rate_emb / drop_rate_attn / drop_rate_shortcut），替代原来统一的 drop_rate。
# 除了 dropout 概率的来源不同外，模型结构、前向传播逻辑与张量形状都与原版 GPTModel 完全一致。
import torch.nn as nn
from gpt import MultiHeadAttention, LayerNorm, FeedForward


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate_attn"], # NEW: dropout for multi-head attention
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        # 中文：这里的 drop_shortcut 使用的是 drop_rate_shortcut，专门用于残差连接处的 dropout，
        # 与注意力内部的 dropout（drop_rate_attn）相互独立。
        self.drop_shortcut = nn.Dropout(cfg["drop_rate_shortcut"])

    def forward(self, x):
        # Shortcut connection for attention block
        # 中文：输入 x 形状为 [batch_size, num_tokens, emb_size]，先保存一份用于残差相加。
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)  # 中文：对注意力子层输出做 dropout（概率为 drop_rate_shortcut）
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)  # 中文：前馈子层输出同样复用 drop_shortcut（drop_rate_shortcut）
        x = x + shortcut  # Add the original input back

        return x  # 中文：输出形状与输入相同 [batch_size, num_tokens, emb_size]


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])  # 中文：词嵌入表，形状 [vocab_size, emb_dim]
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])  # 中文：位置嵌入表，形状 [context_length, emb_dim]
        self.drop_emb = nn.Dropout(cfg["drop_rate_emb"]) # NEW: dropout for embedding layers

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])  # 中文：堆叠 n_layers 个 TransformerBlock

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)  # 中文：输出投影，形状 [emb_dim, vocab_size]

    def forward(self, in_idx):
        # 中文：in_idx 形状为 [batch_size, seq_len]，元素是 token id（整数）。
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)  # 中文：形状 [batch_size, seq_len, emb_dim]
        # 中文：这里用到的全局名 torch 是在下一个单元 `import torch` 之后才在笔记本全局命名空间中出现的；
        # 由于 Python 函数体内的全局名在“调用时”才查找（而不是定义时），只要真正调用 forward 之前
        # torch 已被导入，这里就不会报错——这是本单元一个值得留意的细节，但不是错误，无需修改。
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))  # 中文：形状 [seq_len, emb_dim]
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]  中文：广播相加，融合词义与位置信息
        x = self.drop_emb(x)  # 中文：对嵌入结果做 dropout（概率为 drop_rate_emb）
        x = self.trf_blocks(x)  # 中文：依次通过 n_layers 个 TransformerBlock，形状保持 [batch_size, seq_len, emb_dim]
        x = self.final_norm(x)
        logits = self.out_head(x)  # 中文：形状 [batch_size, seq_len, vocab_size]，每个位置对词表中每个 token 的预测分数
        return logits


In [ ]:
# 中文说明：设置随机种子以保证权重初始化可复现，然后用上面重新定义的 GPTModel
# （带有拆分后的三个 dropout 参数）和 GPT_CONFIG_124M 配置构造最终模型实例。
import torch

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
